# Time-resolved Go vs No-Go distance: model A vs model B

Pick one checkpoint from each of two models (and a task). Both models are run on the *same* simulated trials. Following the paper's definition: each unit's activity is Z-scored, only responsive units are kept (`responsive_criteria.json`), and the single-neuron distance is |r̄_Go − r̄_NoGo| of the class-mean responses. Curves show its mean ± SEM across units at every time bin; epoch values average each unit's responses over the epoch first. Epoch means are compared across units with Welch's t-test (units of two different networks are unpaired).

In [5]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

sys.path.insert(0, os.getcwd())
from neuron_discrimination import time_resolved_distance
%matplotlib inline

In [6]:
# ── set up ────────────────────────────────────────────────────────────────
env_file = Path.cwd().parent / '.env'
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if '=' in line and not line.startswith('#'):
            key, _, val = line.partition('=')
            os.environ.setdefault(key.strip(), val.strip())

results_dir = Path(os.environ.get('TONE_RESULTS_DIR', ''))
if not results_dir:
    sys.exit('TONE_RESULTS_DIR is not set. Add it to .env or export it.')

# the two checkpoints to compare: label, model dir, checkpoint step
A = dict(label='HCT',
         model_dir=results_dir / 'harmonic/harmonic_0_lr5e-05_bs128_noise0.5_nrnn128',
         step=1000)
B = dict(label='PT-HCT',
         model_dir=results_dir / 'transfer_pure_to_harmonic/context+sensory/harmonic/harmonic_0_lr5e-05_bs128_noise0.5_nrnn128',
         step=1000)

TASK = 'harmonic'    # stimulus set both models are tested on
N_TRIALS = 512
SEED = 0             # same seed -> both models see identical trials

# epochs (ms from stimulus onset) for the summary bars
EPOCHS = {'pre-alternation': (0, 60), 'first alternation': (60, 120), 'sustained': (120, 300)}

T_MIN_MS = 0         # time curves start here (ms from stimulus onset)
T_MAX_MS = 300       # time curves are cut off here (ms from stimulus onset)

In [7]:
ra = time_resolved_distance(A['model_dir'], A['step'], task=TASK, n_trials=N_TRIALS, seed=SEED)
rb = time_resolved_distance(B['model_dir'], B['step'], task=TASK, n_trials=N_TRIALS, seed=SEED)
assert np.array_equal(ra['t_ms'], rb['t_ms']), 'models have different trial timing'
t = ra['t_ms']

# single-neuron distance |mean Go - mean No-Go|: mean and SEM across units, per time step
pa, pb = ra['per_neuron'], rb['per_neuron']          # (T, n_units)
mean_a, sem_a = pa.mean(1), pa.std(1, ddof=1) / np.sqrt(pa.shape[1])
mean_b, sem_b = pb.mean(1), pb.std(1, ddof=1) / np.sqrt(pb.shape[1])
# units of two different networks are not paired, so the difference SEM is unpaired
diff, sem_diff = mean_b - mean_a, np.sqrt(sem_a**2 + sem_b**2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2), gridspec_kw={'width_ratios': [2, 2, 1.3]})
on, off = ra['stim_ms']
vis = (t >= T_MIN_MS) & (t <= T_MAX_MS)    # plot only T_MIN_MS..T_MAX_MS

ax = axes[0]
for lab, step, m, s in [(A['label'], A['step'], mean_a, sem_a),
                        (B['label'], B['step'], mean_b, sem_b)]:
    ax.plot(t[vis], m[vis], label=f'{lab} (step {step})')
    ax.fill_between(t[vis], (m - s)[vis], (m + s)[vis], alpha=0.25)
ax.set(xlabel='time from stimulus onset (ms)',
       ylabel='single-neuron distance (mean ± SEM)',
       title=f'Go vs No-Go, {TASK} stimuli  (n = {pa.shape[1]}, {pb.shape[1]} units)')
ax.legend()

ax = axes[1]
ax.plot(t[vis], diff[vis], color='k')
ax.fill_between(t[vis], (diff - sem_diff)[vis], (diff + sem_diff)[vis], color='k', alpha=0.2)
ax.axhline(0, color='0.5', lw=0.8)
ax.set(xlabel='time from stimulus onset (ms)', ylabel='difference (mean ± SEM)',
       title=f"{B['label']} − {A['label']}")

for ax in axes[:2]:
    ax.axvspan(max(on, T_MIN_MS), min(off, T_MAX_MS), color='0.92', zorder=0)
    for (e0, e1) in EPOCHS.values():
        if T_MIN_MS < e1 < T_MAX_MS:
            ax.axvline(e1, color='0.7', lw=0.6, ls=':')
    ax.set_xlim(T_MIN_MS, T_MAX_MS)

# epoch summary: single-neuron distance |r_Go - r_NoGo| of each unit's epoch-averaged
# Go / No-Go responses (average over the epoch first, then take the difference)
names = list(EPOCHS)
rows, ea_all, eb_all = [], [], []
for name, (e0, e1) in EPOCHS.items():
    m = (t >= e0) & (t < e1)
    ea = np.abs(ra['diff'][m].mean(0))                # one value per unit
    eb = np.abs(rb['diff'][m].mean(0))
    tt = ttest_ind(eb, ea, equal_var=False)
    rows.append({'epoch': name, A['label']: ea.mean(), B['label']: eb.mean(),
                 'diff (B−A)': eb.mean() - ea.mean(), 't': tt.statistic, 'p (Welch)': tt.pvalue})
    ea_all.append(ea); eb_all.append(eb)

ax = axes[2]
x = np.arange(len(names))
sem = lambda v: v.std(ddof=1) / np.sqrt(len(v))
ax.bar(x - 0.2, [v.mean() for v in ea_all], 0.4, yerr=[sem(v) for v in ea_all],
       capsize=3, label=A['label'])
ax.bar(x + 0.2, [v.mean() for v in eb_all], 0.4, yerr=[sem(v) for v in eb_all],
       capsize=3, label=B['label'])
# statistical significance markers
for i, row in enumerate(rows):
    if row['p (Welch)'] < 0.05:
        ax.text(i, max([v.mean() for v in ea_all + eb_all]) + 0.02, '*', ha='center', va='bottom', color='red')
    else:
        ax.text(i, max([v.mean() for v in ea_all + eb_all]) + 0.02, 'n.s.', ha='center', va='bottom', color='gray')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha='right')
ax.set(ylabel='mean single-neuron distance (± SEM)', title='epoch means')
ax.legend()

fig.tight_layout()
plt.show()
pd.DataFrame(rows).set_index('epoch')